# What does a new cell cost?

This experiment asks whether a local boundary-event model can predict births per boundary read, including the effects of later rule letters. See [the protocol](README.md) and [the append-only log](NOTEBOOK.md). **Scaffold only: no scientific measurements or fitted models exist here.**

The working assumption is approximate Markov closure of a small local state. Births are not assumed independent regeneration points. The parent experiment motivates locality; it does not establish this closure. Model and pilot choices must be frozen before main measurement.

## Provenance and design
The following cell reads only committed design files under this experiment's results directory, verifies their hashes and checks assignment invariants. Counts below describe a planned cohort, not measured ants. It runs from the repository root or any descendant directory.

In [ ]:
import csv, hashlib, json, subprocess
from collections import Counter, defaultdict
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Cargo.toml').exists() and (p / 'experiments').is_dir())
EXP = ROOT / 'experiments/2026-10-20-c-law-renewal'
inputs = [EXP / 'results/design/families.csv', EXP / 'results/design/provenance.json']
for p in inputs:
    subprocess.run(['git', 'ls-files', '--error-unmatch', str(p.relative_to(ROOT))], cwd=ROOT, check=True, capture_output=True)
    subprocess.run(['git', 'diff', '--exit-code', 'HEAD', '--', str(p.relative_to(ROOT))], cwd=ROOT, check=True, capture_output=True)
    print(hashlib.sha256(p.read_bytes()).hexdigest(), p.relative_to(ROOT))
provenance = json.loads(inputs[1].read_text())
assert hashlib.sha256(inputs[0].read_bytes()).hexdigest() == provenance['manifest_sha256']
print('Source revision:', provenance['base_revision'])
print('Notebook revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip())
print('Sampling seed:', provenance['sampling_seed'])
rows = list(csv.DictReader(inputs[0].open()))
families = defaultdict(list)
for row in rows:
    families[row['family']].append(row)
def necklace(rule):
    mirror = rule.translate(str.maketrans('RL', 'LR'))
    return min(r[i:] + r[:i] for r in (rule, mirror) for i in range(len(r)))
assert len({necklace(r['rule']) for r in rows}) == len(rows)
for family in families.values():
    assert len(family) == 3 and len({r['split'] for r in family}) == 1
    assert {r['arm'] for r in family} == {'base', 'early_swap', 'late_swap'}
    assert len({r['rule'].count('R') for r in family}) == 1
    for r in family:
        assert len(r['rule']) - len(r['rule'].lstrip('R')) == int(r['lead'])
print('Planned families by split:', dict(sorted(Counter(f[0]['split'] for f in families.values()).items())))
print('Planned rules:', len(rows))
print('Scientific results: none; instrument, calibration and pilot pending.')

## Planned method and controls
Train transitions and birth rewards only on training families. Derive validation c from predicted event weights, never from validation event frequencies. Compare against training-fitted lead and survival-kernel baselines. Check paired early/late swaps and independently predicted first, lag-4, other-short and long-read budgets.

Exact reference-path accounting, synthetic reward chains, rotor and random-walk calibration precede an excluded pilot. Report family bootstrap uncertainty and between-interior-seed spread. G0–G3 and P1–P4 in the README define the draft gates and scientific tests; none has been evaluated.

## Results, limits and next steps
There are no scientific results. A planned manifest does not demonstrate local closure, derive the empirical coefficients, or explain kappa. The 3x3 state may be too sparse; long return history may defeat a local model. Censoring and depleted bands can bias apparent costs.

Next: implement and oracle-test ordered events, calibrate estimators, run only the excluded pilot, then append a frozen model and run recipe before measuring the main cohort. Replace this section by appended, provenance-linked observations after those stages, retaining the original predictions in the README.